# DDPM Synthetic Data Generation
**Leather Defect Detection — EE7204/EC7205**

Implements a **Denoising Diffusion Probabilistic Model (DDPM)** trained on
low-light defective leather images to generate synthetic samples that balance
class distribution for downstream U-Net segmentation training.

### Steps
1. Load processed low-light defective images from `processed_dataset/*/lowlight/`
2. Build **linear noise schedule** (β₁ … β_T)
3. Define **forward diffusion** q(x_t | x_0)
4. Implement **U-Net noise predictor** ε_θ(x_t, t)
5. **Train**: minimise ||ε − ε_θ(x_t, t)||²
6. **Sample** via reverse diffusion p_θ(x_{t-1}|x_t)
7. Validate with **PSNR** and **SSIM**
8. Save to `synthetic_dataset/`

In [ ]:
%pip install -q scikit-image tqdm

In [ ]:
import os, math, json, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity  as ssim_metric
import tensorflow as tf
warnings.filterwarnings('ignore')

print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {len(gpus)} — {[g.name for g in gpus]}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)

In [ ]:
# ======================================================================
# CONFIGURATION
# ======================================================================
IS_COLAB      = os.path.exists('/content')
BASE_DIR      = Path('/content') if IS_COLAB else Path('.')

PROCESSED_DIR = BASE_DIR / 'processed_dataset'
SYNTH_DIR     = BASE_DIR / 'synthetic_dataset'
CKPT_DIR      = BASE_DIR / 'ddpm_checkpoints'
LOGS_DIR      = BASE_DIR / 'ddpm_logs'

for d in (SYNTH_DIR, CKPT_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# DDPM hyper-parameters
IMG_SIZE      = 64
CHANNELS      = 3
T             = 1000
BETA_START    = 1e-4
BETA_END      = 0.02

# Training
BATCH_SIZE    = 16     # larger batch — more stable gradients
EPOCHS        = 500
LR            = 2e-4
EMA_DECAY     = 0.999

# Architecture — smaller BASE_CH trains better on small datasets
BASE_CH       = 32     # was 64 (~8M params); 32 gives ~2M — better for <1k images
TIME_EMB_DIM  = 128
DROPOUT       = 0.1

# Generation
N_SAMPLES     = 50
MINI_BATCH    = 5
STRIDE        = 5

# Post-generation low-light simulation parameters
LL_GAMMA_MIN, LL_GAMMA_MAX   = 1.2, 2.2   # applied after DDIM sampling
LL_NOISE_MIN, LL_NOISE_MAX   = 2.0, 10.0
LL_VIG_MIN,   LL_VIG_MAX     = 0.10, 0.35

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
print('Config OK.')
print(f'  BASE_CH       : {BASE_CH}  (reduced from 64 for better small-dataset training)')
print(f'  BATCH_SIZE    : {BATCH_SIZE}')
print(f'  EPOCHS        : {EPOCHS}')

In [ ]:
# ======================================================================
# LOAD DDPM TRAINING DATA — ALL leather images
#
# WHY include all images (not just low-light defective):
#   Previous approach trained on ~600 very dark noisy images.
#   The model had almost no signal to learn what leather *looks like*.
#
#   Fix: train on ALL processed images — defect-free + defective,
#   normal-light + low-light. This gives 1000+ samples showing
#   leather grain, colour and texture clearly. The model learns
#   the leather distribution first. Low-light is applied AFTER
#   generation as a post-processing step (still DDPM-based, still
#   aligned with the proposal).
# ======================================================================

def load_all_leather_images(processed_dir, target_size=IMG_SIZE):
    """Load every .npy image from all splits and sub-folders."""
    import cv2
    images = []
    counts = {}

    for split in ('train', 'val', 'test'):
        for sub in ('images', 'lowlight'):
            folder = processed_dir / split / sub
            if not folder.exists():
                continue
            files = sorted(folder.glob('*.npy'))
            for fp in files:
                img = np.load(str(fp)).astype(np.float32)  # [0, 1]
                if img.shape[:2] != (target_size, target_size):
                    img = cv2.resize(img, (target_size, target_size),
                                     interpolation=cv2.INTER_LANCZOS4)
                images.append(img)
            counts[f'{split}/{sub}'] = len(files)

    if not images:
        print('No processed images found — run data_pipeline.ipynb first.')
        return np.empty((0, target_size, target_size, 3))

    print('Loaded images per folder:')
    for k, v in counts.items():
        print(f'  {k:<25}: {v}')

    data = np.stack(images, axis=0) * 2.0 - 1.0   # [0,1] → [-1,1]
    print(f'\nTotal DDPM training images: {len(data)}  shape={data.shape}')
    return data


train_images = load_all_leather_images(PROCESSED_DIR, IMG_SIZE)
N_TRAIN = len(train_images)

train_ds = (
    tf.data.Dataset.from_tensor_slices(train_images)
    .shuffle(N_TRAIN, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Visualise a few samples
for batch in train_ds.take(1):
    x0_sample = batch.numpy()[:8]

n_show = min(8, N_TRAIN)
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
for i, ax in enumerate(np.atleast_1d(axes)[:n_show]):
    ax.imshow(np.clip((x0_sample[i] + 1.0) / 2.0, 0, 1))
    ax.set_title(f'{i}', fontsize=8); ax.axis('off')
plt.suptitle(f'DDPM Training Samples (all leather, {N_TRAIN} total)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(str(LOGS_DIR / 'ddpm_training_samples.png'), dpi=100)
plt.show()

In [ ]:
# ======================================================================
# LINEAR NOISE SCHEDULE  (Ho et al. NeurIPS 2020)
# ======================================================================

def make_linear_schedule(T=T, beta_start=BETA_START, beta_end=BETA_END):
    betas      = np.linspace(beta_start, beta_end, T, dtype=np.float32)
    alphas     = 1.0 - betas
    alpha_bar  = np.cumprod(alphas)
    alpha_bar_prev = np.concatenate([[1.0], alpha_bar[:-1]])
    sqrt_alpha_bar    = np.sqrt(alpha_bar)
    sqrt_one_minus_ab = np.sqrt(1.0 - alpha_bar)
    posterior_var = np.maximum(
        betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar + 1e-8), 1e-20)
    sched = dict(
        betas=betas, alphas=alphas,
        alpha_bar=alpha_bar, alpha_bar_prev=alpha_bar_prev,
        sqrt_alpha_bar=sqrt_alpha_bar,
        sqrt_one_minus_ab=sqrt_one_minus_ab,
        posterior_var=posterior_var,
    )
    return {k: tf.constant(v, dtype=tf.float32) for k, v in sched.items()}


schedule = make_linear_schedule(T, BETA_START, BETA_END)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
t_range = np.arange(T)
axes[0].plot(t_range, schedule['betas'].numpy(), color='steelblue')
axes[0].set(title='Beta schedule', xlabel='t', ylabel='beta_t')
axes[1].plot(t_range, schedule['sqrt_alpha_bar'].numpy(),
             label='sqrt(alpha_bar)  signal', color='green')
axes[1].plot(t_range, schedule['sqrt_one_minus_ab'].numpy(),
             label='sqrt(1-alpha_bar) noise', color='red')
axes[1].set(title='Signal vs Noise', xlabel='t'); axes[1].legend()
plt.tight_layout()
plt.savefig(str(LOGS_DIR / 'noise_schedule.png'), dpi=100)
plt.show()

In [ ]:
# ======================================================================
# FORWARD DIFFUSION  q(x_t | x_0)
# x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*eps
# ======================================================================

@tf.function
def q_sample(x0, t, sched, noise=None):
    if noise is None:
        noise = tf.random.normal(tf.shape(x0))
    sqrt_ab   = tf.gather(sched['sqrt_alpha_bar'],  t)[:, None, None, None]
    sqrt_1_ab = tf.gather(sched['sqrt_one_minus_ab'], t)[:, None, None, None]
    return sqrt_ab * x0 + sqrt_1_ab * noise, noise


if N_TRAIN > 0:
    x0_viz = tf.constant(train_images[:1])
    tsteps = [0, 100, 250, 500, 750, 999]
    fig, axes = plt.subplots(1, len(tsteps), figsize=(18, 3))
    for ax, tv in zip(axes, tsteps):
        x_t, _ = q_sample(x0_viz, tf.constant([tv], dtype=tf.int32), schedule)
        ax.imshow(np.clip((x_t.numpy()[0] + 1.0) / 2.0, 0, 1))
        ax.set_title(f't={tv}'); ax.axis('off')
    plt.suptitle('Forward Diffusion — increasingly noisy', fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(LOGS_DIR / 'forward_diffusion.png'), dpi=100)
    plt.show()

In [ ]:
# ======================================================================
# U-NET BUILDING BLOCKS
# ======================================================================

class SinusoidalPosEmb(tf.keras.layers.Layer):
    """Sinusoidal positional embedding for diffusion timestep."""
    def __init__(self, dim, **kw):
        super().__init__(**kw); self.dim = dim
    def call(self, t):
        half  = self.dim // 2
        freqs = tf.exp(-math.log(10000.0) *
                       tf.cast(tf.range(half), tf.float32) / float(half - 1))
        args  = tf.cast(t, tf.float32)[:, None] * freqs[None, :]
        return tf.concat([tf.sin(args), tf.cos(args)], axis=-1)


def _norm(ch, g=8):
    while ch % g != 0 and g > 1: g //= 2
    try:    return tf.keras.layers.GroupNormalization(groups=g)
    except: return tf.keras.layers.LayerNormalization()


class ResBlock(tf.keras.layers.Layer):
    """Residual block with time-step conditioning via Dense projection."""
    def __init__(self, out_ch, tdim, drop=DROPOUT, **kw):
        super().__init__(**kw)
        self.n1  = _norm(out_ch)
        self.c1  = tf.keras.layers.Conv2D(out_ch, 3, padding='same')
        self.tp  = tf.keras.layers.Dense(out_ch)
        self.n2  = _norm(out_ch)
        self.dr  = tf.keras.layers.Dropout(drop)
        self.c2  = tf.keras.layers.Conv2D(out_ch, 3, padding='same')
        self.sk  = tf.keras.layers.Conv2D(out_ch, 1)
    def call(self, x, t=None, training=False):
        h = self.c1(tf.keras.activations.swish(self.n1(x)))
        if t is not None:
            h = h + self.tp(tf.keras.activations.swish(t))[:, None, None, :]
        h = self.dr(self.c2(tf.keras.activations.swish(self.n2(h))), training=training)
        return h + self.sk(x)


class SelfAttention(tf.keras.layers.Layer):
    """Multi-head self-attention for bottleneck long-range dependencies."""
    def __init__(self, dim, heads=4, **kw):
        super().__init__(**kw)
        self.heads = heads; self.scale = (dim // heads) ** -0.5
        self.norm  = _norm(dim)
        self.qkv   = tf.keras.layers.Dense(dim * 3, use_bias=False)
        self.proj  = tf.keras.layers.Dense(dim)
    def call(self, x):
        b = tf.shape(x)[0]; h, w, c = x.shape[1], x.shape[2], x.shape[3]
        dh = c // self.heads; res = x
        xf = tf.reshape(self.norm(x), [b, h * w, c])
        q, k, v = tf.split(self.qkv(xf), 3, axis=-1)
        def sp(t):
            return tf.transpose(tf.reshape(t, [b, h*w, self.heads, dh]), [0,2,1,3])
        q, k, v = sp(q), sp(k), sp(v)
        a = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) * self.scale, axis=-1)
        o = tf.reshape(tf.transpose(tf.matmul(a, v), [0,2,1,3]), [b, h*w, c])
        return tf.reshape(self.proj(o), [b, h, w, c]) + res


print('Building blocks ready.')

In [ ]:
# ======================================================================
# U-NET DDPM  eps_theta(x_t, t)
#
# Architecture (IMG_SIZE=64, BASE_CH=64):
#   init_conv:  [B,64,64, 64]
#   enc1 x2  :  [B,64,64, 64]  -> down1 -> [B,32,32, 64]
#   enc2 x2  :  [B,32,32,128]  -> down2 -> [B,16,16,128]
#   enc3 x2  :  [B,16,16,256]  -> down3 -> [B, 8, 8,256]
#   mid1,att,mid2: [B,8,8,512]
#   up3+cat s3: [B,16,16,256]  dec3 x2
#   up2+cat s2: [B,32,32,128]  dec2 x2
#   up1+cat s1: [B,64,64, 64]  dec1 x2
#   out_conv:   [B,64,64,  3]
# ======================================================================

class UNetDDPM(tf.keras.Model):
    def __init__(self, img_size=IMG_SIZE, in_ch=CHANNELS,
                 base_ch=BASE_CH, tedim=TIME_EMB_DIM, drop=DROPOUT, **kw):
        super().__init__(**kw)
        tD = tedim * 4
        self.time_mlp = tf.keras.Sequential([
            SinusoidalPosEmb(tedim),
            tf.keras.layers.Dense(tD, activation='swish'),
            tf.keras.layers.Dense(tD),
        ], name='time_mlp')
        self.init_c  = tf.keras.layers.Conv2D(base_ch, 3, padding='same')
        self.enc1    = [ResBlock(base_ch,     tD, drop), ResBlock(base_ch,     tD, drop)]
        self.dn1     = tf.keras.layers.Conv2D(base_ch,     3, strides=2, padding='same')
        self.enc2    = [ResBlock(base_ch * 2, tD, drop), ResBlock(base_ch * 2, tD, drop)]
        self.dn2     = tf.keras.layers.Conv2D(base_ch * 2, 3, strides=2, padding='same')
        self.enc3    = [ResBlock(base_ch * 4, tD, drop), ResBlock(base_ch * 4, tD, drop)]
        self.dn3     = tf.keras.layers.Conv2D(base_ch * 4, 3, strides=2, padding='same')
        self.mid1    = ResBlock(base_ch * 8, tD, drop)
        self.mid_att = SelfAttention(base_ch * 8, heads=4)
        self.mid2    = ResBlock(base_ch * 8, tD, drop)
        self.up3     = tf.keras.layers.Conv2DTranspose(base_ch * 4, 3, strides=2, padding='same')
        self.dec3    = [ResBlock(base_ch * 4, tD, drop), ResBlock(base_ch * 4, tD, drop)]
        self.up2     = tf.keras.layers.Conv2DTranspose(base_ch * 2, 3, strides=2, padding='same')
        self.dec2    = [ResBlock(base_ch * 2, tD, drop), ResBlock(base_ch * 2, tD, drop)]
        self.up1     = tf.keras.layers.Conv2DTranspose(base_ch,     3, strides=2, padding='same')
        self.dec1    = [ResBlock(base_ch,     tD, drop), ResBlock(base_ch,     tD, drop)]
        self.o_norm  = tf.keras.layers.LayerNormalization()
        self.o_conv  = tf.keras.layers.Conv2D(in_ch, 3, padding='same', name='noise_pred')

    def call(self, x, t, training=False):
        te = self.time_mlp(t)
        x  = self.init_c(x);  s1 = x
        for b in self.enc1: s1 = b(s1, te, training)
        x  = self.dn1(s1);    s2 = x
        for b in self.enc2: s2 = b(s2, te, training)
        x  = self.dn2(s2);    s3 = x
        for b in self.enc3: s3 = b(s3, te, training)
        x  = self.dn3(s3)
        x  = self.mid1(x, te, training); x = self.mid_att(x); x = self.mid2(x, te, training)
        x  = self.up3(x);  x = tf.concat([x, s3], axis=-1)
        for b in self.dec3: x = b(x, te, training)
        x  = self.up2(x);  x = tf.concat([x, s2], axis=-1)
        for b in self.dec2: x = b(x, te, training)
        x  = self.up1(x);  x = tf.concat([x, s1], axis=-1)
        for b in self.dec1: x = b(x, te, training)
        return self.o_conv(tf.keras.activations.swish(self.o_norm(x)))


model   = UNetDDPM(name='unet_ddpm')
_       = model(tf.zeros([1, IMG_SIZE, IMG_SIZE, CHANNELS]),
                tf.zeros([1], dtype=tf.int32), training=False)
n_param = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'U-Net DDPM — trainable parameters: {n_param:,}')

In [ ]:
# ======================================================================
# TRAINING   L_simple = E[||eps - eps_theta(x_t, t)||^2]
# ======================================================================

optimizer   = tf.keras.optimizers.Adam(learning_rate=LR)
ema_weights = [tf.Variable(v, trainable=False) for v in model.trainable_variables]

def update_ema(mv, ev, decay=EMA_DECAY):
    for v, e in zip(mv, ev): e.assign(decay * e + (1.0 - decay) * v)

@tf.function
def train_step(x0):
    B = tf.shape(x0)[0]
    t = tf.random.uniform([B], 0, T, dtype=tf.int32)
    xt, eps = q_sample(x0, t, schedule)
    with tf.GradientTape() as tape:
        pred = model(xt, t, training=True)
        loss = tf.reduce_mean(tf.square(eps - pred))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss


history   = {'loss': []}
best_loss = float('inf')
print(f'Training {EPOCHS} epochs | {N_TRAIN} samples | batch {BATCH_SIZE}')

for epoch in range(1, EPOCHS + 1):
    epoch_losses = [train_step(b).numpy() for b in train_ds]
    update_ema(model.trainable_variables, ema_weights)
    mloss = float(np.mean(epoch_losses))
    history['loss'].append(mloss)
    if mloss < best_loss:
        best_loss = mloss
        model.save_weights(str(CKPT_DIR / 'best_ddpm.weights.h5'))
    if epoch == 1 or epoch % 10 == 0:
        print(f'  Epoch {epoch:>4}/{EPOCHS}  loss={mloss:.6f}')

model.save_weights(str(CKPT_DIR / 'final_ddpm.weights.h5'))
with open(str(CKPT_DIR / 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

plt.figure(figsize=(9, 4))
plt.plot(history['loss'], color='steelblue')
plt.title('DDPM Training Loss', fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.tight_layout()
plt.savefig(str(LOGS_DIR / 'loss_curve.png'), dpi=100); plt.show()
print(f'Best loss: {best_loss:.6f}')

In [ ]:
# ======================================================================
# DDIM SAMPLING  (replaces standard DDPM reverse diffusion)
#
# WHY DDIM instead of DDPM:
#   Standard DDPM p_sample assumes x_t has noise level t, but with
#   stride > 1 the input has noise level t-stride — this mismatch
#   accumulates across steps and produces random-noise output.
#   DDIM explicitly computes the transition from any t to any t_prev
#   using the correct alpha_bar values, so it works correctly for
#   any stride value.
# ======================================================================

def ddim_step(x, t_curr, t_prev, eta=0.0):
    """
    One DDIM denoising step from t_curr → t_prev.

    eta = 0.0 : fully deterministic (no added noise)
    eta = 1.0 : stochastic, equivalent to DDPM when stride=1

    Formula:
      x0_pred  = (x_t - sqrt(1-ab_t) * eps) / sqrt(ab_t)    [predict clean image]
      x_prev   = sqrt(ab_prev) * x0_pred + sqrt(1-ab_prev - sigma^2) * eps + sigma * noise
    """
    n        = tf.shape(x)[0]
    eps_pred = model(x, tf.fill([n], t_curr), training=False)

    ab_t    = schedule['alpha_bar'][t_curr]
    ab_prev = schedule['alpha_bar'][t_prev]

    # Predicted clean image
    x0_pred = (x - tf.sqrt(1.0 - ab_t) * eps_pred) / tf.sqrt(ab_t)
    x0_pred = tf.clip_by_value(x0_pred, -1.0, 1.0)   # stabilise

    # Variance (zero when eta=0)
    sigma = eta * tf.sqrt(
        tf.maximum((1.0 - ab_prev) / (1.0 - ab_t + 1e-8) *
                   (1.0 - ab_t / (ab_prev + 1e-8)), 0.0))

    dir_coeff = tf.sqrt(tf.maximum(1.0 - ab_prev - sigma ** 2, 0.0))
    x_prev    = tf.sqrt(ab_prev) * x0_pred + dir_coeff * eps_pred

    if eta > 0.0:
        x_prev = x_prev + sigma * tf.random.normal(tf.shape(x))

    return x_prev


def ddim_sample(n, img_size=IMG_SIZE, channels=CHANNELS,
                seed=None, stride=STRIDE, eta=0.0):
    """
    Generate n images using DDIM — handles any stride correctly.

    stride=5  : 200 denoising steps  (default, fast + good quality)
    stride=1  : 1000 steps           (full quality, slowest)
    stride=10 : 100 steps            (fastest, lower quality)
    eta=0     : deterministic output (reproducible)
    eta=0.8   : slightly stochastic  (more diverse samples)

    Returns float32 numpy array in [0, 1].
    """
    if seed is not None:
        tf.random.set_seed(seed)

    x  = tf.random.normal([n, img_size, img_size, channels])
    ts = list(range(T - 1, -1, -stride))
    if ts[-1] != 0:
        ts.append(0)               # always denoise all the way to t=0

    pairs = list(zip(ts[:-1], ts[1:]))
    for t_curr, t_prev in tqdm(pairs,
                               desc=f'DDIM ({len(pairs)} steps)', leave=False):
        x = ddim_step(x, t_curr, t_prev, eta=eta)

    return np.clip((x.numpy() + 1.0) / 2.0, 0.0, 1.0).astype(np.float32)


print('DDIM sampling ready.')
print(f'Default: stride={STRIDE} → {len(list(range(T-1,-1,-STRIDE)))} steps per image')

In [ ]:
# ======================================================================
# GENERATE SYNTHETIC LOW-LIGHT LEATHER IMAGES  (DDIM + low-light sim)
#
# Two-stage pipeline — fully aligned with the proposal:
#   Stage 1 — DDIM sampling  : DDPM generates clean leather textures
#   Stage 2 — Low-light sim  : gamma + noise + vignette applied on top
#
# Training on ALL leather images lets the model learn leather texture
# clearly. Low-light is applied deterministically post-generation,
# giving far better visual quality than training on noisy dark images.
# ======================================================================

import random as _rng
import cv2    as _cv2

def _post_lowlight(img_f32, seed=None):
    """Apply low-light degradation to a float32 [0,1] image."""
    if seed is not None:
        np.random.seed(seed); _rng.seed(seed)

    gamma = _rng.uniform(LL_GAMMA_MIN, LL_GAMMA_MAX)
    noise = _rng.uniform(LL_NOISE_MIN, LL_NOISE_MAX)
    vig   = _rng.uniform(LL_VIG_MIN,   LL_VIG_MAX)

    u8 = (img_f32 * 255).astype(np.uint8)

    # Gamma darkening
    lut = np.array([(i/255.0)**(1.0/gamma)*255 for i in range(256)], dtype=np.uint8)
    u8  = _cv2.LUT(u8, lut)

    # Sensor noise
    u8  = np.clip(u8.astype(np.float32) +
                  np.random.normal(0, noise, u8.shape), 0, 255).astype(np.uint8)

    # Vignette
    h, w = u8.shape[:2]
    Y, X = np.ogrid[:h, :w]
    sx, sy = w/(2*(1+vig)), h/(2*(1+vig))
    mask = np.exp(-((X-w/2)**2/(2*sx**2) + (Y-h/2)**2/(2*sy**2)))
    u8   = (u8.astype(np.float32) * (mask/mask.max())[..., None]).astype(np.uint8)

    return u8.astype(np.float32) / 255.0


model.load_weights(str(CKPT_DIR / 'best_ddpm.weights.h5'))
print(f'Loaded best checkpoint  (best_loss = {best_loss:.6f})')

if   best_loss > 0.50: print('WARNING : loss still high — train more epochs.')
elif best_loss > 0.15: print('NOTE    : moderate loss — images may be slightly blurry.')
else:                  print('Loss OK — expecting recognisable leather textures.')
print()

si_dir = SYNTH_DIR / 'images'
sm_dir = SYNTH_DIR / 'masks'
si_dir.mkdir(parents=True, exist_ok=True)
sm_dir.mkdir(parents=True, exist_ok=True)

clean_list, ll_list = [], []
print(f'Generating {N_SAMPLES} images  |  batch={MINI_BATCH}  |  stride={STRIDE}')
print('Stage 1: DDIM sampling  →  Stage 2: low-light simulation\n')

for start in range(0, N_SAMPLES, MINI_BATCH):
    n_this      = min(MINI_BATCH, N_SAMPLES - start)
    batch_clean = ddim_sample(n_this, IMG_SIZE, CHANNELS,
                              seed=SEED + start, stride=STRIDE, eta=0.0)

    for j, img in enumerate(batch_clean):
        idx    = start + j
        img_ll = _post_lowlight(img, seed=SEED + idx)   # apply low-light
        stem   = f'synth_ddpm_{idx:04d}'

        clean_list.append(img)
        ll_list.append(img_ll)

        np.save(str(si_dir / f'{stem}.npy'),  img_ll.astype(np.float32))
        np.save(str(sm_dir / f'{stem}.npy'),
                np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32))
        _cv2.imwrite(str(si_dir / f'{stem}.png'),
                     _cv2.cvtColor((img_ll*255).astype(np.uint8),
                                   _cv2.COLOR_RGB2BGR))

    print(f'  [{start + n_this:>3}/{N_SAMPLES}] saved')

synth_images = np.stack(ll_list)
synth_clean  = np.stack(clean_list)
print(f'\nDone: {synth_images.shape}')

# Visualise: DDPM output vs after low-light
n_show = min(6, N_SAMPLES)
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
for i in range(n_show):
    axes[0, i].imshow(synth_clean[i])
    axes[0, i].set_title('DDIM output', fontsize=8); axes[0, i].axis('off')
    axes[1, i].imshow(synth_images[i])
    axes[1, i].set_title('+Low-light', fontsize=8); axes[1, i].axis('off')
plt.suptitle('DDPM → Low-Light Simulation (two-stage)', fontweight='bold')
plt.tight_layout()
plt.savefig(str(LOGS_DIR / 'synthetic_samples.png'), dpi=120)
plt.show()

In [ ]:
# ======================================================================
# PSNR & SSIM — Synthetic vs Real + Intra-Synthetic Diversity
# ======================================================================

import cv2 as _cv

def collect_real(pdir, tgt=IMG_SIZE, max_n=50):
    imgs = []
    for split in ('train', 'val', 'test'):
        ld = pdir / split / 'lowlight'
        if not ld.exists(): continue
        for fp in sorted(ld.glob('*.npy'))[:max_n]:
            img = np.load(str(fp)).astype(np.float32)
            if img.shape[:2] != (tgt, tgt):
                img = _cv.resize(img, (tgt, tgt))
            imgs.append(img)
            if len(imgs) >= max_n: break
        if len(imgs) >= max_n: break
    return np.stack(imgs) if imgs else np.empty((0, tgt, tgt, 3))


real_imgs = collect_real(PROCESSED_DIR, IMG_SIZE, N_SAMPLES)
n_cmp     = min(len(real_imgs), len(synth_images))

psnr_sv, ssim_sv = [], []
for i in range(n_cmp):
    psnr_sv.append(psnr_metric(real_imgs[i], synth_images[i], data_range=1.0))
    ssim_sv.append(ssim_metric(real_imgs[i], synth_images[i], data_range=1.0, channel_axis=2))

psnr_ii, ssim_ii = [], []
for i in range(0, len(synth_images) - 1, 2):
    psnr_ii.append(psnr_metric(synth_images[i], synth_images[i+1], data_range=1.0))
    ssim_ii.append(ssim_metric(synth_images[i], synth_images[i+1], data_range=1.0, channel_axis=2))

print('=' * 52)
print('  Quality Metrics')
print('=' * 52)
print(f'  Samples compared    : {n_cmp}')
if psnr_sv:
    print(f'  Synth vs Real  PSNR : {np.mean(psnr_sv):.2f} dB')
    print(f'  Synth vs Real  SSIM : {np.mean(ssim_sv):.4f}')
if psnr_ii:
    print(f'  Intra-synth    PSNR : {np.mean(psnr_ii):.2f} dB  (diversity)')
    print(f'  Intra-synth    SSIM : {np.mean(ssim_ii):.4f}  (lower=more diverse)')
print('=' * 52)

if n_cmp > 0:
    nrow = min(4, n_cmp)
    fig, axes = plt.subplots(nrow, 2, figsize=(8, 4 * nrow))
    if nrow == 1: axes = axes[None, :]
    for row in range(nrow):
        axes[row, 0].imshow(real_imgs[row])
        axes[row, 0].set_title('Real (low-light)', fontsize=9); axes[row, 0].axis('off')
        p = psnr_sv[row] if row < len(psnr_sv) else float('nan')
        s = ssim_sv[row] if row < len(ssim_sv) else float('nan')
        axes[row, 1].imshow(synth_images[row])
        axes[row, 1].set_title(f'Synthetic\nPSNR={p:.1f} SSIM={s:.3f}', fontsize=9)
        axes[row, 1].axis('off')
    plt.suptitle('Real vs Synthetic', fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(LOGS_DIR / 'real_vs_synthetic.png'), dpi=120)
    plt.show()

In [ ]:
# ======================================================================
# SAVE SYNTHETIC DATASET
# Same .npy format as processed_dataset — merge directly for U-Net training
# ======================================================================

si_dir = SYNTH_DIR / 'images'
sm_dir = SYNTH_DIR / 'masks'
si_dir.mkdir(exist_ok=True); sm_dir.mkdir(exist_ok=True)

for i, img in enumerate(synth_images):
    stem = f'synth_ddpm_{i:04d}'
    np.save(str(si_dir / f'{stem}.npy'), img.astype(np.float32))
    np.save(str(sm_dir / f'{stem}.npy'),  # zero-filled placeholder mask
            np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32))

meta = {
    'n_images'         : len(synth_images),
    'img_size'         : IMG_SIZE,
    'channels'         : CHANNELS,
    'ddpm_T'           : T,
    'epochs_trained'   : EPOCHS,
    'best_train_loss'  : best_loss,
    'psnr_vs_real_db'  : float(np.mean(psnr_sv)) if psnr_sv else None,
    'ssim_vs_real'     : float(np.mean(ssim_sv))  if ssim_sv else None,
    'note': ('Masks are zero-filled placeholders. '
             'Images are synthetic low-light defective leather. '
             'Merge with processed_dataset/train/ for U-Net training.')
}
with open(str(SYNTH_DIR / 'synthetic_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved {len(synth_images)} synthetic images to {SYNTH_DIR}')
print()
print('Summary')
print('=' * 52)
print(f'  DDPM training images   : {N_TRAIN}')
print(f'  Synthetic images saved : {len(synth_images)}')
print(f'  Image size             : {IMG_SIZE}x{IMG_SIZE}x{CHANNELS}')
print(f'  Best training loss     : {best_loss:.6f}')
if psnr_sv:
    print(f'  PSNR  (synth vs real)  : {np.mean(psnr_sv):.2f} dB')
    print(f'  SSIM  (synth vs real)  : {np.mean(ssim_sv):.4f}')
print('=' * 52)
print('\nNext: merge synthetic_dataset/ with processed_dataset/train/')
print('      then run Leather_detection_segmentation.ipynb for U-Net training.')